<a href="https://colab.research.google.com/github/Rakshit-Gupta-77/Machine-Learning-Journey/blob/main/categorical_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import libraries

We use **NumPy** and **Pandas** for data handling, and scikit-learn encoders for converting categorical values into numbers.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

## 2. Create a small customer dataset

The original notebook expected a `customer.csv` file and later used variables such as `X_train` and `y_train` before defining them.

To make this notebook runnable by itself, we create the same kind of categorical data directly.

- `Education`: ordered categories — School < UG < PG
- `Income`: ordered categories — Poor < Average < Good
- `Purchased`: target column
- `Age`: numerical feature

In [2]:
df = pd.DataFrame({
    "CustomerID": range(1, 11),
    "Age": [22, 25, 28, 31, 35, 38, 42, 24, 30, 45],
    "Income": ["Poor", "Average", "Good", "Average", "Good",
               "Poor", "Good", "Average", "Poor", "Good"],
    "Education": ["School", "UG", "PG", "UG", "PG",
                  "School", "PG", "UG", "School", "PG"],
    "Purchased": ["No", "No", "Yes", "Yes", "Yes",
                  "No", "Yes", "No", "No", "Yes"]
})

df

,CustomerID,Age,Income,Education,Purchased
0,1,22,Poor,School,No
1,2,25,Average,UG,No
2,3,28,Good,PG,Yes
3,4,31,Average,UG,Yes
4,5,35,Good,PG,Yes
5,6,38,Poor,School,No
6,7,42,Good,PG,Yes
7,8,24,Average,UG,No
8,9,30,Poor,School,No
9,10,45,Good,PG,Yes


## 3. Inspect the data

`sample(5)` displays five random rows. `head()` displays the first five rows.

In [3]:
df.sample(5, random_state=42)

,CustomerID,Age,Income,Education,Purchased
8,9,30,Poor,School,No
1,2,25,Average,UG,No
5,6,38,Poor,School,No
0,1,22,Poor,School,No
7,8,24,Average,UG,No


In [4]:
df.head()

,CustomerID,Age,Income,Education,Purchased
0,1,22,Poor,School,No
1,2,25,Average,UG,No
2,3,28,Good,PG,Yes
3,4,31,Average,UG,Yes
4,5,35,Good,PG,Yes


## 4. Select features and target

The original notebook removed the first two columns using `df.iloc[:, 2:]`.

Here we explicitly select the columns so the purpose is clear.

- `X` contains input features.
- `y` contains the target.

In [5]:
X = df[["Age", "Income", "Education"]].copy()
y = df["Purchased"].copy()

X.head(), y.head()

(   Age   Income Education
 0   22     Poor    School
 1   25  Average        UG
 2   28     Good        PG
 3   31  Average        UG
 4   35     Good        PG,
 0     No
 1     No
 2    Yes
 3    Yes
 4    Yes
 Name: Purchased, dtype: object)

## 5. Split the data

We split the dataset into training and testing sets.

**Important:** encoders should generally be fitted using the training data only. This prevents information from the test set from leaking into the training process.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (8, 3)
X_test shape : (2, 3)
y_train shape: (8,)
y_test shape : (2,)


## 6. Ordinal Encoding

`OrdinalEncoder` converts categorical **input features** into numbers.

It is appropriate when the categories have a meaningful order.

For example:

`Poor < Average < Good`

and

`School < UG < PG`

We explicitly provide these orders so the encoder does not assign arbitrary numerical values.

In [7]:
oe = OrdinalEncoder(
    categories=[
        ["Poor", "Average", "Good"],   # Income
        ["School", "UG", "PG"]         # Education
    ]
)

### Encode the categorical training columns

Only `Income` and `Education` are categorical. `Age` is already numerical, so it should not be passed to `OrdinalEncoder`.

We use `.fit()` on the training categories and `.transform()` to convert them.

In [8]:
categorical_cols = ["Income", "Education"]

oe.fit(X_train[categorical_cols])

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

X_train_encoded[categorical_cols] = oe.transform(X_train[categorical_cols])
X_test_encoded[categorical_cols] = oe.transform(X_test[categorical_cols])

X_train_encoded

,Age,Income,Education
6,42,2.0,2.0
8,30,0.0,0.0
1,25,1.0,1.0
0,22,0.0,0.0
3,31,1.0,1.0
2,28,2.0,2.0
4,35,2.0,2.0
5,38,0.0,0.0


## 7. Check the learned category order

`oe.categories_` shows the category order learned by the encoder.

The resulting mapping is:

- Income: `Poor → 0`, `Average → 1`, `Good → 2`
- Education: `School → 0`, `UG → 1`, `PG → 2`

In [9]:
oe.categories_

[array(['Poor', 'Average', 'Good'], dtype=object),
 array(['School', 'UG', 'PG'], dtype=object)]

## 8. Label Encoding

`LabelEncoder` is mainly intended for encoding the **target variable (`y`)**.

It converts class labels such as:

`No`, `Yes`

into numerical labels.

Do not use `LabelEncoder` as the normal choice for multiple input feature columns.

In [10]:
le = LabelEncoder()

le.fit(y_train)

print("Classes:", le.classes_)

Classes: ['No' 'Yes']


### Transform the target

After fitting on `y_train`, use the same encoder to transform both `y_train` and `y_test`.

This keeps the mapping consistent between training and testing data.

In [11]:
y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

print("Encoded y_train:", y_train_encoded)
print("Encoded y_test :", y_test_encoded)

Encoded y_train: [1 0 0 0 1 1 1 0]
Encoded y_test : [0 1]


## 9. See the complete mappings

`LabelEncoder` stores the class order in `classes_`.

For example, if:

`le.classes_ = ['No', 'Yes']`

then:

- `No → 0`
- `Yes → 1`

In [12]:
print("LabelEncoder classes:", le.classes_)

LabelEncoder classes: ['No' 'Yes']


## 10. Decode labels back to their original values

`inverse_transform()` converts encoded values back into the original labels.

In [13]:
decoded = le.inverse_transform(y_train_encoded)

print("Encoded :", y_train_encoded)
print("Decoded :", decoded)

Encoded : [1 0 0 0 1 1 1 0]
Decoded : ['Yes' 'No' 'No' 'No' 'Yes' 'Yes' 'Yes' 'No']


## 11. `fit()` vs `transform()` vs `fit_transform()`

### `fit()`
Learns the mapping from the data.

### `transform()`
Uses an already learned mapping to convert data.

### `fit_transform()`
Performs both operations in one step.

For a train/test workflow, a common pattern is:

```python
encoder.fit(X_train)
X_train_encoded = encoder.transform(X_train)
X_test_encoded = encoder.transform(X_test)
```

Do **not** fit the encoder separately on the test set.

## 12. Final encoded dataset

At this point:

- `Age` remains numerical.
- `Income` and `Education` have been ordinal encoded.
- `Purchased` has been label encoded.

These numerical values can now be supplied to many machine-learning algorithms that require numerical input.

In [14]:
print("X_train encoded:")
display(X_train_encoded)

print("\ny_train encoded:")
display(y_train_encoded)

X_train encoded:


,Age,Income,Education
6,42,2.0,2.0
8,30,0.0,0.0
1,25,1.0,1.0
0,22,0.0,0.0
3,31,1.0,1.0
2,28,2.0,2.0
4,35,2.0,2.0
5,38,0.0,0.0



y_train encoded:


array([1, 0, 0, 0, 1, 1, 1, 0])

## Quick revision

| Encoder | Common use | Example |
|---|---|---|
| `OrdinalEncoder` | Categorical input features with an order | Poor < Average < Good |
| `LabelEncoder` | Target/class labels | No → 0, Yes → 1 |

### Key rule
**Feature columns → `OrdinalEncoder` (when appropriate)**  
**Target labels → `LabelEncoder`**

Also remember: **fit on training data, then transform both training and test data using the same fitted encoder.**